**문서 형식 확장**

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day04" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day04"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"

MODELS = ROOT.parent / "models"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [2]:
if not SAMPLES.is_dir():
    print('재료 폴더가 없습니다:', SAMPLES)
    print('9/17 노트북 00 번의 복사 셀을 한 번 더 돌리세요.')
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]
    root_files = [p for p in SAMPLES.iterdir() if p.is_file()]
    fmt_files = [p for p in FORMATS.iterdir() if p.is_file()]
    print(f'재료 파일 : {len(all_files)}개')
    print(f'  samples 바로 아래 : {len(root_files)}개 · _formats 안 : {len(fmt_files)}개')
    print('_formats 안의 재료 :')
    for p in sorted(FORMATS.glob('sample_*')):
        print('  ' + p.name + ('   ← 오후 03 번' if p.suffix == '.pdf' else ''))


재료 파일 : 30개
  samples 바로 아래 : 24개 · _formats 안 : 6개
_formats 안의 재료 :
  sample_guide.md
  sample_memo.txt
  sample_notice.pptx
  sample_page.html
  sample_scanned.pdf   ← 오후 03 번
  sample_table.xlsx


**다른 형식이 현재 local_parser 에서 파싱이 되는지 확인**

In [3]:
from app.core.exceptions import ValidationFailed
from app.rag.local_parsers import parse_local
today = ['sample_table.xlsx', 'sample_notice.pptx', 'sample_memo.txt', 'sample_guide.md', 'sample_page.html']
for name in today:
    p = FORMATS / name
    try:
        doc = parse_local(p)
        if doc is None:
            print(f'{p.name:<20}-> 파서가 도중에 터졌습니다 (로그를 보세요)')
        else:
            print(f'{p.name:<20}-> 블록 {len(doc.blocks)}개')
    except ValidationFailed as e:
        print(f'{p.name:<20}-> ValidationFailed: {e.message}  | detail={e.detail}')


sample_table.xlsx   -> 블록 1개
sample_notice.pptx  -> 블록 1개
sample_memo.txt     -> ValidationFailed: 지원하지 않는 형식입니다: .txt  | detail=핸들러를 추가하면 지원할 수 있습니다
sample_guide.md     -> ValidationFailed: 지원하지 않는 형식입니다: .md  | detail=핸들러를 추가하면 지원할 수 있습니다
sample_page.html    -> ValidationFailed: 지원하지 않는 형식입니다: .html  | detail=핸들러를 추가하면 지원할 수 있습니다


**파싱할 것이 없는 파일**

In [4]:
from app.core.exceptions import ValidationFailed
from app.rag.local_parsers import parse_local

memo = FORMATS / 'sample_memo.txt'
text = memo.read_text(encoding='utf-8')

print(f'[그냥 읽으면]  {len(text)}자')
print(text)
print('[parse_local 에 넣으면]')
try:
    parse_local(memo)
except ValidationFailed as e:
    print(f'ValidationFailed: {e} | detail={e.detail}')


[그냥 읽으면]  71자
출장 정산 안내

9월부터 출장 정산은 그룹웨어에서만 접수합니다.
종이 서류는 받지 않습니다.

문의: 인사총무팀 김지원 과장

[parse_local 에 넣으면]
ValidationFailed: 지원하지 않는 형식입니다: .txt | detail=핸들러를 추가하면 지원할 수 있습니다


In [5]:
_PLAIN = {'.txt', '.md'}
_HTML = {'.html', '.htm'}
_LOCAL_ONLY = {'.hwp', '.hwpx'}

def which(name, use_upstage=False):
    ext = pathlib.Path(name).suffix.lower()
    if ext in _PLAIN:
        return '_parse_plain'
    if ext in _HTML:
        return '_parse_html'
    if use_upstage and ext not in _LOCAL_ONLY:
        return 'upstage'
    return 'parse_local'
    
names = ['sample_memo.txt', 'sample_guide.md', 'sample_page.html', 'sample_table.xlsx', 'sample_notice.pptx', 'DOC-HR-014_국내출장_여비_규정_v2.0.docx', 'DOC-HR-014_국내출장_여비_규정_v2.0.hwpx']
print('[use_upstage=False]')

for n in names:
    print(f'  {n:<40} -> {which(n)}')
print()
print('[use_upstage=True]')

for n in names:
    before, after = (which(n), which(n, use_upstage=True))
    mark = '(바뀐다)' if before != after else '(그대로)'
    print(f'  {n:<40} -> {after:<14} {mark}')

[use_upstage=False]
  sample_memo.txt                          -> _parse_plain
  sample_guide.md                          -> _parse_plain
  sample_page.html                         -> _parse_html
  sample_table.xlsx                        -> parse_local
  sample_notice.pptx                       -> parse_local
  DOC-HR-014_국내출장_여비_규정_v2.0.docx          -> parse_local
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx          -> parse_local

[use_upstage=True]
  sample_memo.txt                          -> _parse_plain   (그대로)
  sample_guide.md                          -> _parse_plain   (그대로)
  sample_page.html                         -> _parse_html    (그대로)
  sample_table.xlsx                        -> upstage        (바뀐다)
  sample_notice.pptx                       -> upstage        (바뀐다)
  DOC-HR-014_국내출장_여비_규정_v2.0.docx          -> upstage        (바뀐다)
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx          -> parse_local    (그대로)


In [6]:
mem = FORMATS / 'sample_memo.txt'

text = mem.read_text(encoding='utf-8')
print(f'[그냥 읽으면]  {len(text)}자')

[그냥 읽으면]  71자


**플레인 파서**

In [7]:
memo = FORMATS / "sample_memo.txt"

text = memo.read_text(encoding="utf-8", errors="replace") 

parts = [s.strip() for s in text.split("\n\n") if s.strip()]

for i, s in enumerate(parts, 1):
    print(f"[{i}] {len(s)}자 : {s}")

print(f"블록 {len(parts)}개")

[1] 8자 : 출장 정산 안내
[2] 42자 : 9월부터 출장 정산은 그룹웨어에서만 접수합니다.
종이 서류는 받지 않습니다.
[3] 16자 : 문의: 인사총무팀 김지원 과장
블록 3개


In [8]:
guide = FORMATS / "sample_guide.md"

text = guide.read_text(encoding="utf-8", errors="replace")
parts = [s.strip() for s in text.split("\n\n") if s.strip()]

for i, s in enumerate(parts, 1):
    print(f"[{i}] {len(s)}자 : {s}")

print(f"블록 {len(parts)}개")

[1] 13자 : # 출장 신청 빠른 안내
[2] 8자 : ## 1. 신청
[3] 23자 : 출장 3일 전까지 그룹웨어에서 신청합니다.
[4] 8자 : ## 2. 승인
[5] 39자 : 팀장 승인 후 확정됩니다. 5일 초과 출장은 본부장 승인이 필요합니다.
[6] 8자 : ## 3. 정산
[7] 27자 : 복귀 후 7일 이내에 영수증을 첨부해 정산합니다.
블록 7개


**HTML 파서**

In [9]:
# 나쁜 예시
import re

page = FORMATS / "sample_page.html"
html = page.read_text(encoding="utf-8", errors="replace")

# 패턴으로 태그 형태 변경: 
flat = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", html)).strip()

print(flat)

blocks = [s.strip() for s in flat.split("\n\n") if s.strip()]
print(f"블록 {len(blocks)}개")  # -> 하나의 블럭으로 합쳐짐 (나쁜 예시)

사내 공지 추석 연휴 근무 안내 연휴 기간 중 당직 근무자는 아래와 같습니다. 날짜 당직자 연락처 10/03 김민준 내선 2201 10/04 이서연 내선 2202
블록 1개


In [10]:
# script, style 태그 묶어서 지우기
raw = re.sub('<(script|style)[^>]*>.*?</\\1>', ' ', html, flags=re.S | re.I)

# 표 모아놓고 동일한 패턴으로 태그 삭제
tables = re.findall('<table.*?</table>', raw, flags=re.S | re.I)
body = re.sub('<table.*?</table>', ' ', raw, flags=re.S | re.I)

# 남은 태그 지우고 공백 정리
body = re.sub('\\s+', ' ', re.sub('<[^>]+>', ' ', body)).strip()

# 떼어둔 표마다 태그를 지워 표 블럭으로 만들기
blocks = [('조항', page.name, body)] if body else []
for i, table in enumerate(tables, 1):
    cleaned = re.sub('\\s+', ' ', re.sub('<[^>]+>', ' ', table)).strip()
    blocks.append(('표', f'{page.name} · 표{i}', cleaned))
# 카운트
total = sum((len(t) for _, _, t in blocks))
print(f'[좋은 예] 블록 {len(blocks)} · 표 {len(tables)} · 쪽 1 · 글자 {total}자')

for n, (kind, locator, t) in enumerate(blocks, 1):
    print(f'  [{n}] {kind}  {locator:<26} {len(t):3d}자  {t[:40]}')

[좋은 예] 블록 2 · 표 1 · 쪽 1 · 글자 89자
  [1] 조항  sample_page.html            43자  사내 공지 추석 연휴 근무 안내 연휴 기간 중 당직 근무자는 아래와 같습
  [2] 표  sample_page.html · 표1       46자  날짜 당직자 연락처 10/03 김민준 내선 2201 10/04 이서연 내


*HTML 안에는 &<와 같은 기능을 갖는 문자들이 본문에 포함되어있는 경우들이 있다. (제외 대상)*

In [11]:
import importlib
from app.rag import parser
importlib.reload(parser)
targets = [FORMATS / 'sample_memo.txt', FORMATS / 'sample_guide.md', FORMATS / 'sample_page.html', FORMATS / 'sample_table.xlsx', FORMATS / 'sample_notice.pptx', SAMPLES / 'DOC-HR-014_국내출장_여비_규정_v2.0.docx']
for p in targets:
    try:
        doc = parser.parse(p)
        print(f'{p.name:<40} 블록 {len(doc.blocks):3d}  표 {doc.table_count}  쪽 {doc.page_count}')
    except ValidationFailed as e:
        print(f'{p.name:<40} ValidationFailed: {e}')


sample_memo.txt                          블록   3  표 0  쪽 1
sample_guide.md                          블록   7  표 0  쪽 1
sample_page.html                         블록   2  표 1  쪽 1
sample_table.xlsx                        블록   1  표 1  쪽 1
sample_notice.pptx                       블록   1  표 0  쪽 1
DOC-HR-014_국내출장_여비_규정_v2.0.docx          블록 233  표 7  쪽 1


In [12]:
from openpyxl import load_workbook    # workbook 엑셀 파일 worksheet 파일 안 시트 

XLSX = FORMATS / "sample_table.xlsx"

wb = load_workbook(str(XLSX), data_only=True)

print("시트 목록 : ", wb.sheetnames) # 시트 이름들 목록 

ws = wb[wb.sheetnames[0]]  # 시트 이름으로 시트 한장 추출 
print("시트 이름 : ", ws.title) 

# 셀 객체 아닌, 셀 안의 값들을 튜플로, 행단위로 추출 
for row in ws.iter_rows(values_only=True):
    print(row)

시트 목록 :  ['출장비 집계']
시트 이름 :  출장비 집계
('부서', '1분기', '2분기', '3분기')
('인프라사업부 2팀', 4200000, 3800000, 5100000)
('구매팀', 1200000, 1400000, 1100000)
('보안팀', 900000, 1000000, 1300000)


In [13]:
import importlib
from app.rag import local_parsers
importlib.reload(local_parsers)

doc = local_parsers.parse_xlsx(FORMATS / 'sample_table.xlsx')
chars = sum((len(b.text) for b in doc.blocks))
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count} · 글자 {chars}자')
first = doc.blocks[0]
print('  kind    :', first.kind)
print('  locator :', first.locator)
for line in first.text.split('\n')[:3]:
    print(line)


블록 1 · 표 1 · 쪽 1 · 글자 161자
  kind    : 표
  locator : 출장비 집계
| 부서 | 1분기 | 2분기 | 3분기 |
|---|---|---|---|
| 인프라사업부 2팀 | 4200000 | 3800000 | 5100000 |


**병합 셀**

In [14]:
from openpyxl import Workbook

wb = Workbook()
ws = wb.active

ws.title = '부서별 집행액'
ws.append(['구분', '1분기', '2분기'])
ws.append(['인프라사업부', 4200000, 3800000])
ws.append(['인프라사업부', 1200000, 1400000])
ws.append(['구매팀', 900000, 1000000])
ws.merge_cells('A2:A3')
MERGED = SANDBOX / 'merged.xlsx'
wb.save(str(MERGED))

print(f'만든 파일 : {MERGED.relative_to(ROOT)} ({MERGED.stat().st_size:,} 바이트)')

만든 파일 : sandbox\w5\day04\merged.xlsx (5,022 바이트)


**PPTX 파서**

In [15]:
# 잘못된 예시 
from pptx import Presentation

PPTX = FORMATS / "sample_notice.pptx"
prs = Presentation(str(PPTX))   # PPT 파일 열기 

lines = []
for slide in prs.slides:
    for shape in slide.shapes:
        if shape.has_text_frame and shape.text.strip():
            lines.append(shape.text.strip())

body = "\n".join(lines)
print(f"슬라이드 : {len(prs.slides)}장, {len(body)}자")
print(body)


슬라이드 : 1장, 70자
출장 여비 규정 개정 안내
일비 25,000 → 30,000원
광역시 숙박비 상한 70,000원 신설
시행 2025-07-01


In [16]:
# 좋은 예시 
from pptx import Presentation

PPTX = FORMATS / "sample_notice.pptx"
prs = Presentation(str(PPTX))   # PPT 파일 열기 

lines = []
for slide in prs.slides:
    for shape in slide.shapes:
        if shape.has_text_frame and shape.text.strip():
            lines.append(shape.text.strip())
    # 슬라이드 기준, 발표자 노트 있는지 체크 
    if slide.has_notes_slide:
        # 발표자 노트의 글자만 긁어오기 
        note = slide.notes_slide.notes_text_frame.text.strip() 
        if note:
            lines.append(f"[발표자 노트] {note}")

body = "\n".join(lines)
print(f"슬라이드 : {len(prs.slides)}장, {len(body)}자")
print(body)


슬라이드 : 1장, 99자
출장 여비 규정 개정 안내
일비 25,000 → 30,000원
광역시 숙박비 상한 70,000원 신설
시행 2025-07-01
[발표자 노트] 인사총무팀 설명용 발표 노트입니다.


In [17]:
print("HANDLERS 현재:", list(local_parsers.HANDLERS))
print()

# parse_xlsx 를 직접 부르지 않는다. parse_local 로 불러 '분기표를 거치는지' 를 본다.
for name in ("sample_table.xlsx", "sample_notice.pptx"):
    doc = local_parsers.parse_local(FORMATS / name)
    chars = sum(len(b.text) for b in doc.blocks)
    print(f"{name:<19} 블록 {len(doc.blocks)} - 표 {doc.table_count} - "
          f"쪽 {doc.page_count} - {chars:>3}자")

HANDLERS 현재: ['.docx', '.pdf', '.hwpx', '.xlsx', '.pptx']

sample_table.xlsx   블록 1 - 표 1 - 쪽 1 - 161자
sample_notice.pptx  블록 1 - 표 0 - 쪽 1 -  99자


In [19]:
from app.rag import parser
from app.rag.chunker import chunk, summarize
importlib.reload(parser)
FIVE = ['sample_memo.txt', 'sample_guide.md', 'sample_page.html', 'sample_table.xlsx', 'sample_notice.pptx']
for name in FIVE:
    doc = parser.parse(FORMATS / name)
    chunks = chunk(doc)
    print(f'{name:<19} 블록 {len(doc.blocks):>2}  표 {doc.table_count}  →  {summarize(chunks)}')

sample_memo.txt     블록  3  표 0  →  조항 단위 1 + 표 단위 0 = 1 청크
sample_guide.md     블록  7  표 0  →  조항 단위 1 + 표 단위 0 = 1 청크


AttributeError: 'tuple' object has no attribute 'kind'